# 🧭 01 - Apache Polaris REST Catalog Setup

### 📌 Objective
In this notebook, we initialize and authenticate with the **Apache Polaris REST Catalog**.

### 💡 Why an Iceberg REST Catalog?
- Traditional metastores (like Hive Metastore) require heavy Java dependencies and expose low-level storage credentials.
- The **Apache Iceberg REST Catalog** specification standardizes metadata operations over simple, stateless HTTP APIs.
- Polaris serves as the single centralized source of truth for **PySpark**, **Trino**, and **Snowflake**.

In [ ]:
import os
import requests
from dotenv import load_dotenv

load_dotenv()

# When running inside the Jupyter container, Polaris is reached at http://polaris:8181
POLARIS_URL = os.getenv("POLARIS_URL", "http://polaris:8181")
CLIENT_ID = os.getenv("POLARIS_CLIENT_ID", "polaris")
CLIENT_SECRET = os.getenv("POLARIS_CLIENT_SECRET", "polaris")
WAREHOUSE_NAME = "icelake"

print(f"Target Polaris URL: {POLARIS_URL}")

### Step 1: Obtain OAuth2 Bearer Token
Polaris uses OAuth2 client credentials to secure catalog interactions.

In [ ]:
resp = requests.post(
    f"{POLARIS_URL}/api/catalog/v1/oauth/tokens",
    data={
        "grant_type": "client_credentials",
        "client_id": CLIENT_ID,
        "client_secret": CLIENT_SECRET,
        "scope": "PRINCIPAL_ROLE:ALL"
    }
)
resp.raise_for_status()
token = resp.json()["access_token"]
print("✅ Successfully obtained OAuth2 Bearer Token!")
print(f"Token (truncated): {token[:40]}...")

### Step 2: Create the `icelake` Catalog
We create an **INTERNAL** catalog pointing to our Garage S3 bucket: `s3://icelake/`.
We set `"stsUnavailable": True` because Garage S3 is a standalone cluster that doesn't use AWS STS token exchange.

In [ ]:
headers = {
    "Authorization": f"Bearer {token}",
    "Content-Type": "application/json"
}

catalog_payload = {
    "name": WAREHOUSE_NAME,
    "type": "INTERNAL",
    "properties": {
        "default-base-location": "s3://icelake/",
        "s3.endpoint": "http://garage:3900",
        "s3.path-style-access": "true"
    },
    "storageConfigInfo": {
        "storageType": "S3",
        "allowedLocations": ["s3://icelake/", "s3://icelake"],
        "stsUnavailable": True,
        "endpoint": "http://garage:3900",
        "pathStyleAccess": True
    }
}

resp = requests.post(
    f"{POLARIS_URL}/api/management/v1/catalogs",
    headers=headers,
    json=catalog_payload
)
print(f"Catalog Response: {resp.status_code}")
if resp.status_code in [200, 201]:
    print(f"✅ Catalog '{WAREHOUSE_NAME}' registered successfully!")
elif resp.status_code == 409:
    print(f"ℹ️ Catalog '{WAREHOUSE_NAME}' already exists.")
else:
    print(resp.text)

### Step 3: Create the `wikipedia` Namespace
Namespaces in Iceberg function like relational database schemas.

In [ ]:
ns_payload = {"namespace": ["wikipedia"]}

resp = requests.post(
    f"{POLARIS_URL}/api/catalog/v1/{WAREHOUSE_NAME}/namespaces",
    headers=headers,
    json=ns_payload
)
print(f"Namespace Response: {resp.status_code}")
if resp.status_code in [200, 201]:
    print("✅ Namespace 'wikipedia' created successfully!")
elif resp.status_code == 409:
    print("ℹ️ Namespace 'wikipedia' already exists.")
else:
    print(resp.text)

### Step 4: List Registered Namespaces

In [ ]:
resp = requests.get(
    f"{POLARIS_URL}/api/catalog/v1/{WAREHOUSE_NAME}/namespaces",
    headers=headers
)
print("Namespaces in icelake:", resp.json())